# 02. Treinamento e Fine-Tuning do Detector YOLOv8 de Placas Veiculares

Este notebook está **100% pronto para execução no Google Colab com GPU gratuita**.

### Fluxo Automatizado:
1. **Ambiente:** Instalação das dependências e checagem da GPU.
2. **Dados:** Download automático do dataset de placas (`keremberke/license-plate-object-detection`) do Hugging Face e conversão direta para formato YOLO.
3. **Treino:** Fine-tuning do modelo leve `yolov8n.pt` para detecção de placas.
4. **Avaliação:** Cálculo das métricas de teste (mAP@0.5, precisão e recall).
5. **Exportação ONNX:** Conversão com `opset=12` e `simplify=True` para inferência rápida offline em CPU.
6. **Verificação & Download:** Checagem dos tensores de saída e download do arquivo `best.onnx` diretamente para sua máquina.

---
## 1. Instalação de Dependências e Checagem da GPU
*Certifique-se de que a GPU está ativada em: `Ambiente de Execução` -> `Alterar tipo de ambiente de execução` -> `T4 GPU`.*

In [ ]:
!pip install -q ultralytics datasets onnx onnxruntime

import torch
print('Versão do PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('✅ GPU detectada:', torch.cuda.get_device_name(0))
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU no menu Ambiente de Execução!')

---
## 2. Download Automático do Dataset e Conversão para Formato YOLO
Baixa o dataset público de detecção de placas e organiza as pastas em `images/` e `labels/` com coordenadas normalizadas.

In [ ]:
import os
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

print("Baixando dataset diretamente via Parquet do Hugging Face...")
base_url = (
    "https://huggingface.co/datasets/"
    "keremberke/license-plate-object-detection/"
    "resolve/refs%2Fconvert%2Fparquet/full"
)

# Carrega os arquivos Parquet diretamente (100% compatível com as versões recentes)
ds = load_dataset(
    "parquet",
    data_files={
        "train": f"{base_url}/train/0000.parquet",
        "val": f"{base_url}/validation/0000.parquet",
        "test": f"{base_url}/test/0000.parquet",
    }
)

print("Dataset carregado com sucesso:", ds)

base_dir = Path("/content/dataset")
splits = ["train", "val", "test"]

# Dica: deixe None para exportar as 6.176 imagens de treino,
# ou coloque um número (ex: 2000) se quiser acelerar o primeiro teste
LIMITE_TREINO = None

for split in splits:
    img_dir = base_dir / "images" / split
    lbl_dir = base_dir / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    subset = ds[split]
    if split == "train" and LIMITE_TREINO is not None:
        subset = subset.select(range(min(LIMITE_TREINO, len(subset))))

    print(f"Exportando split {split} ({len(subset)} amostras)...")

    for i, item in enumerate(tqdm(subset)):
        img = item["image"]
        w_img, h_img = img.size

        img_name = f"{split}_{i:05d}.jpg"
        img.save(img_dir / img_name, format="JPEG")

        linhas_yolo = []
        for bbox in item["objects"]["bbox"]:
            xmin, ymin, bw, bh = bbox
            # Coordenadas normalizadas para YOLO: cx, cy, w, h
            cx = (xmin + bw / 2.0) / w_img
            cy = (ymin + bh / 2.0) / h_img
            w_norm = bw / w_img
            h_norm = bh / h_img
            linhas_yolo.append(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}")

        lbl_name = f"{split}_{i:05d}.txt"
        with open(lbl_dir / lbl_name, "w", encoding="utf-8") as f:
            f.write("\n".join(linhas_yolo))

# Criação do arquivo de configuração do dataset para o YOLO
yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: placa
"""

with open("/content/placas.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("✅ Dataset estruturado e placas.yaml criado com sucesso!")

---
## 3. Treinamento (Fine-Tuning) com YOLOv8n
Usamos a variante nano (`yolov8n.pt`) para obter alta precisão com custo computacional mínimo para inferência rápida no desktop.

In [ ]:
from ultralytics import YOLO

# Carrega pesos pré-treinados
modelo = YOLO('yolov8n.pt')

# Inicia fine-tuning (ajuste epochs conforme disponibilidade de tempo, ex: 30 a 50)
resultados = modelo.train(
    data='/content/placas.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project='runs/detect',
    name='treino_placas',
    verbose=True
)

---
## 4. Avaliação das Métricas no Conjunto de Teste

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Encontra automaticamente o arquivo best.pt gerado pelo treino
candidatos = list(Path("/content").glob("**/treino_placas*/weights/best.pt"))

if not candidatos:
    raise FileNotFoundError("Arquivo best.pt não encontrado. Verifique se o treino terminou.")

caminho_best_pt = str(candidatos[0])
print("Arquivo best.pt encontrado em:", caminho_best_pt)
melhor_modelo = YOLO(caminho_best_pt)
metricas = melhor_modelo.val(data="/content/placas.yaml", split="test")
print("=" * 50)
print("MÉTRICAS NO CONJUNTO DE TESTE")
print("=" * 50)
print(f"mAP@0.5:      {metricas.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metricas.box.map:.4f}")
print(f"Precisão:     {metricas.box.mp:.4f}")
print(f"Recall:       {metricas.box.mr:.4f}")
print("=" * 50)

---
## 5. Exportação para Formato ONNX
Exporta o modelo otimizado para CPU, eliminando a dependência de PyTorch no aplicativo desktop.

In [ ]:
caminho_onnx = melhor_modelo.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True
)
print('✅ Modelo ONNX exportado em:', caminho_onnx)

---
## 6. Verificação do Modelo com onnxruntime em CPU

In [ ]:
import onnxruntime as ort

# Usa diretamente a variável da célula anterior
sessao = ort.InferenceSession(caminho_onnx, providers=["CPUExecutionProvider"])
entrada = sessao.get_inputs()[0]
saida = sessao.get_outputs()[0]

print("Entrada ONNX:", entrada.name, entrada.shape)
print("Saída ONNX:  ", saida.name, saida.shape)
print("✅ Modelo verificado e 100% compatível!")

---
## 7. Download do Modelo para o Desktop
Ao baixar o arquivo `best.onnx`, renomeie-o para `placas.onnx` e coloque-o na pasta `models/` do projeto no seu computador.

In [ ]:
from google.colab import files

print(f"Iniciando download de: {caminho_onnx}")
files.download(caminho_onnx)